# First steps with DigitalHub: entities and artifacts

This tutorial introduces the basic shape of a DigitalHub entity and shows how to store, register, and download artifacts.

You need a configured DigitalHub environment and the `digitalhub` Python SDK.

## 1. Create a project

A project is the workspace that groups entities. Use your username to create a personal project.

In [ ]:
import getpass
from pathlib import Path

import digitalhub as dh

username = getpass.getuser().lower()
project = dh.get_or_create_project(f"{username}-base")
project

## 2. Anatomy of an entity

An entity has a **kind**, a **name**, a **spec**, and a **status**:

- **kind** says what the entity represents, such as `artifact`.
- **name** is its human-readable, logical name.
- **spec** describes the entity, for example the path to its file.
- **status** is maintained by the platform and describes its state.

Artifacts are versioned: the name identifies the artifact across versions, while its ID identifies one specific version. Creating another version with the same name keeps the same logical entity name but gives the version a new ID.

## 3. Upload a local file

Use `log_artifact` to upload a local file and create its catalog entry. The returned object includes the entity metadata.

In [ ]:
Path("hello.txt").write_text("Hello, DigitalHub!\n")

artifact = project.log_artifact(
    name="hello-file",
    source="hello.txt",
    description="A small example file",
)
artifact

The SDK handles the creation of the entity, the upload of the file in the right repository (with credentials) and updates the status with the backend.

In [ ]:
print("kind:", artifact.kind)
print("name:", artifact.name)
print("spec.path:", artifact.spec.path)
print("status:", artifact.status.state)
print("version ID:", artifact.id)

Log the same name again to create another version. Compare the IDs: the name stays the same, but the version ID changes.

In [ ]:
Path("hello.txt").write_text("Hello again, DigitalHub!\n")
artifact_v2 = project.log_artifact(
    name="hello-file",
    source="hello.txt",
    description="A second version of the example file",
)

print("names:", artifact.name, artifact_v2.name)
print("IDs:", artifact.id, artifact_v2.id)

We can discover and retrieve any previous version from the catalog

In [ ]:
versions = project.get_artifact_versions("hello-file")
for version in versions:
    print(f"Version: {version.id}, created: {version.metadata.created}")

And every file is still available, because every version has its own space in the repository.

In [ ]:
for version in versions:
    print(f"Version: {version.id}, path: {version.spec.path}")

And we can access any version

In [ ]:
first = versions[-1]
print("kind:", first.kind)
print("name:", first.name)
print("created:", first.metadata.created)
print("spec.path:", first.spec.path)
print("status:", first.status.state)
print("version ID:", first.id)

## 4. Register an external file

If a file already has an HTTP URL, register its location instead of uploading a local copy. In this SDK version, `new_artifact` creates the catalog entry from the URL; `log_artifact` is for uploading local files.

In [ ]:
url = "https://raw.githubusercontent.com/scc-digitalhub/atlas-platform-course/main/1-getting-started/data.json"

external_artifact = project.register_artifact(
    name="example-json",
    source=url,
    description="Example file registered from an HTTP URL",
)
external_artifact

In [ ]:
print("kind:", external_artifact.kind)
print("name:", external_artifact.name)
print("created:", external_artifact.metadata.created)
print("spec.path:", external_artifact.spec.path)
print("status:", external_artifact.status.state)
print("version ID:", external_artifact.id)

## 5. Download an artifact

Artifacts contain a static, stable reference to the file(s).

`as_file()` downloads to a temporary location and returns a list of file paths. `download()` saves a copy at the destination you choose.

In [ ]:
temporary_path = artifact.as_file()
temporary_path

In [ ]:
# read from path
with open(temporary_path, "r") as f:
    content = f.read()
content

Download let us specify the destination.

In [ ]:
Path("downloads").mkdir(exist_ok=True)
downloaded_path = artifact.download(
    destination="downloads/hello-copy.txt",
    overwrite=True,
)
downloaded_path

In [ ]:
# read from path
with open(downloaded_path, "r") as f:
    content = f.read()
content

The same works for external resources like `external_artifact`

In [ ]:
downloaded_path = external_artifact.download(
    destination="downloads/data.json",
    overwrite=True
)
downloaded_path
# read from path
with open(downloaded_path, "r") as f:
    content = f.read()
content

## 6. Store a folder

An artifact can also be a directory. Pass the directory as the source to upload its contents together.

In [ ]:
folder = Path("example-folder")
folder.mkdir(exist_ok=True)
(folder / "readme.txt").write_text("Files can be grouped in an artifact folder.\n")
(folder / "notes.txt").write_text("This is a second file.\n")

folder_artifact = project.log_artifact(
    name="example-folder",
    source=str(folder),
)
folder_artifact

We can see the details of files uploaded via `files()`

In [ ]:
for f in folder_artifact.files:
    print(f"file {f.name} at path {f.path}\n\tdetails: {f}")

And we can download back the whole folder

In [ ]:
folder_artifact.download(destination="downloads/example-folder", overwrite=True)

Or single files only

In [ ]:
folder_file = folder_artifact.files[0]
folder_file

In [ ]:
file_path = folder_file.as_file()
file_path
# read from path
with open(file_path, "r") as f:
    content = f.read()
content

## Exercise

1. Check with the console everything we did is consistent. Then download a file via the browser.
2. Upload an artifact (text file) with the console and then download from the SDK. Hint: we can `get` artifacts with the name
3. Upload a new version of the same artifact with a different content, via either console or SDK, and then `get` again. Which *version* do we get back?
